In [1]:
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# ML models
model_lr = pickle.load(open("../models/model_lr.pkl", "rb"))
model_nb = pickle.load(open("../models/model_nb.pkl", "rb"))
model_svm = pickle.load(open("../models/model_svm.pkl", "rb"))

# vectorizer + labels
vectorizer = pickle.load(open("../models/vectorizer.pkl", "rb"))
mlb = pickle.load(open("../models/mlb.pkl", "rb"))

# threshold
threshold = pickle.load(open("../models/threshold.pkl", "rb"))

# embeddings
embeddings = pickle.load(open("../models/embeddings.pkl", "rb"))

# sentence transformer
with open("../models/embedding_model.txt") as f:
    model_name = f.read().strip()

model_st = SentenceTransformer(model_name)

print("✅ All models loaded!")

✅ All models loaded!


In [3]:
df = pd.read_csv("../data/processed_dataset.csv")

# 🔥 IMPORTANT (same as training)
df["text"] = df["titles"] + " " + df["summaries"]

print("Dataset loaded:", df.shape)

Dataset loaded: (135939, 8)


In [4]:
def smart_recommend(query, top_k=10):
    
    # =========================
    # 🔹 Step 1: Prepare query
    # =========================
    query_text = query
    
    # =========================
    # 🔹 Step 2: Predict labels
    # =========================
    vec = vectorizer.transform([query_text])
    
    p1 = model_lr.predict_proba(vec)
    p2 = model_nb.predict_proba(vec)
    p3 = model_svm.decision_function(vec)

    # normalize SVM scores
    p3 = (p3 - p3.min()) / (p3.max() - p3.min())

    probs = (0.5*p1 + 0.3*p2 + 0.2*p3)
    probs = np.power(probs, 1.2)

    idx = np.where(probs[0] >= threshold)[0]

    # ensure at least 2 labels
    if len(idx) < 2:
        idx = np.argsort(probs[0])[-2:]

    labels = [mlb.classes_[i] for i in idx]
    
    
    # =========================
    # 🔹 Step 3: Filter dataset
    # =========================
    mask = df["terms"].apply(lambda x: any(label in x for label in labels))
    filtered_df = df[mask]

    if len(filtered_df) == 0:
        filtered_df = df
        filtered_embeddings = embeddings
    else:
        filtered_embeddings = embeddings[mask.values]
    
    
    # =========================
    # 🔹 Step 4: Semantic similarity
    # =========================
    query_emb = model_st.encode([query_text])
    
    sim = cosine_similarity(query_emb, filtered_embeddings)[0]
    
    
    # =========================
    # 🔹 Step 5: SORT (IMPORTANT 🔥)
    # =========================
    sorted_idx = np.argsort(sim)[::-1]   # highest first
    
    top_idx = sorted_idx[:top_k]
    
    
    # =========================
    # 🔹 Step 6: Prepare results
    # =========================
    results = filtered_df.iloc[top_idx].copy()
    
    # add similarity score
    results["similarity_score"] = sim[top_idx]
    
    
    return {
        "predicted_labels": labels,
        "recommendations": results[[
            "titles",
            "category",
            "authors",
            "first_author",
            "terms",
            "published_date",
            "summaries"
        ]]
    }

In [5]:
# print(smart_recommend("Ai powered tourism"))
# print(smart_recommend("deep learning for medical images"))
print(smart_recommend("AI Powered Tourism"))

{'predicted_labels': ['cs.HC', 'cs.AI'], 'recommendations':                                                   titles  \
4218   Artificial Intelligence Systems applied to tou...   
3627                Artificial Intelligence and Robotics   
4858   Towards Sustainable Artificial Intelligence: A...   
23901  Problematizing AI Omnipresence in Landscape Ar...   
7185   An Artificial Intelligence-based Framework to ...   
4714                  Künstliche Intelligenz, quo vadis?   
3462                  Artificial Intelligence Approaches   
12741  Artificial Intelligence: Powering Human Explor...   
6426                     The AI Index 2022 Annual Report   
21326      Explainable Sustainability for AI in the Arts   

                         category  \
4218      Artificial Intelligence   
3627      Artificial Intelligence   
4858      Artificial Intelligence   
23901     Artificial Intelligence   
7185      Artificial Intelligence   
4714      Artificial Intelligence   
3462      Artificial

In [6]:
queries = [
    "deep learning for image classification",
    "transformer models in NLP",
    "reinforcement learning in robotics"
]

for q in queries:
    res = smart_recommend(q, top_k=5)
    
    print("\n" + "="*60)
    print("Query:", q)
    print("Predicted Labels:", res["predicted_labels"])
    
    display(res["recommendations"])


Query: deep learning for image classification
Predicted Labels: ['cs.LG', 'cs.CV']


,titles,category,authors,first_author,terms,published_date,summaries
28004,Introduction to deep learning,Machine Learning,"['Lihi Shiloh-Perl', 'Raja Giryes']",'Lihi Shiloh-Perl',['cs.LG'],2/29/20,Deep Learning (DL) has made a major impact on ...
51792,Some Improvements on Deep Convolutional Neural...,Computer Vision and Pattern Recognition,['Andrew G. Howard'],'Andrew G. Howard',['cs.CV'],12/19/13,We investigate multiple techniques to improve ...
66059,Flower Categorization using Deep Convolutional...,Computer Vision and Pattern Recognition,"['Ayesha Gurnani', 'Viraj Mavani', 'Vandit Gaj...",'Ayesha Gurnani',['cs.CV'],08-12-2017,We have developed a deep learning network for ...
54697,A Hybrid Deep Learning Approach for Texture An...,Computer Vision and Pattern Recognition,"['Hussein Adly', 'Mohamed Moustafa']",'Hussein Adly',['cs.CV'],3/24/17,Texture classification is a problem that has v...
55377,Classification of Medical Images and Illustrat...,Computer Vision and Pattern Recognition,"['Jianpeng Zhang', 'Yong Xia', 'Qi Wu', 'Yuton...",'Jianpeng Zhang',['cs.CV'],6/28/17,The Classification of medical images and illus...



Query: transformer models in NLP
Predicted Labels: ['q-bio.QM', 'cs.CL']


,titles,category,authors,first_author,terms,published_date,summaries
24407,A Survey on Transformers in NLP with Focus on ...,Computation and Language (Natural Language Pro...,"['Wazib Ansar', 'Saptarsi Goswami', 'Amlan Cha...",'Wazib Ansar',['cs.CL'],5/15/24,The advent of transformers with attention mech...
133463,DistilCamemBERT: a distillation of the French ...,Computation and Language (Natural Language Pro...,"['Cyrile Delestre', 'Abibatou Amar']",'Cyrile Delestre',['cs.CL'],5/23/22,Modern Natural Language Processing (NLP) model...
128737,A Primer on the Inner Workings of Transformer-...,Computation and Language (Natural Language Pro...,"['Javier Ferrando', 'Gabriele Sarti', 'Arianna...",'Javier Ferrando',['cs.CL'],4/30/24,The rapid progress of research aimed at interp...
126457,A Family of Pretrained Transformer Language Mo...,Computation and Language (Natural Language Pro...,"['Dmitry Zmitrovich', 'Alexander Abramov', 'An...",'Dmitry Zmitrovich',['cs.CL'],9/19/23,Transformer language models (LMs) are fundamen...
125075,Transformer models: an introduction and catalog,Computation and Language (Natural Language Pro...,"['Xavier Amatriain', 'Ananth Sankar', 'Jie Bin...",'Xavier Amatriain',['cs.CL'],02-12-2023,In the past few years we have seen the meteori...



Query: reinforcement learning in robotics
Predicted Labels: ['cs.LG', 'cs.RO']


,titles,category,authors,first_author,terms,published_date,summaries
85348,A framework for reinforcement learning with au...,Machine Learning,"['Marcin Szulc', 'Jakub Łyskawa', 'Paweł Wawrz...",'Marcin Szulc',['cs.LG'],09-10-2020,The subject of this paper is reinforcement lea...
37960,Model-based Policy Optimization using Symbolic...,Machine Learning,"['Andrey Gorodetskiy', 'Konstantin Mironov', '...",'Andrey Gorodetskiy',['cs.LG'],7/18/24,The application of learning-based control meth...
98833,Learning Transition Models with Time-delayed C...,Machine Learning,"['Junchi Liang', 'Abdeslam Boularias']",'Junchi Liang',['cs.LG'],08-04-2020,This paper introduces an algorithm for discove...
102934,A survey on policy search algorithms for learn...,Robotics,"['Konstantinos Chatzilygeroudis', 'Vassilis Va...",'Konstantinos Chatzilygeroudis',['cs.RO'],07-06-2018,Most policy search algorithms require thousand...
84527,Gaussian Process Policy Optimization,Machine Learning,"['Ashish Rao', 'Bidipta Sarkar', 'Tejas Naraya...",'Ashish Rao',['cs.LG'],03-02-2020,"We propose a novel actor-critic, model-free re..."
